In [ ]:
import asyncio
import aiohttp
import pandas as pd
import numpy as np
import time
from tqdm.asyncio import tqdm_asyncio

# ======================================
# CONFIG
# ======================================

API_KEY = "hidden"

MAX_CONCURRENT = 10     # Parallel requests — safe for 500 URLs
RETRY_ATTEMPTS = 3      # Retries per failed request
RETRY_BACKOFF  = 2      # Base wait in seconds (doubles each retry)
STRATEGY       = "desktop"  # Keep consistent — do NOT mix desktop/mobile
RUNS_PER_URL   = 3      # Number of measurements per URL for mean/std

# ======================================
# 479 UNIQUE URLS (from top500_websites.txt)
# ======================================

urls = [
    "https://www.abcnews.go.com",
    "https://www.academia.edu",
    "https://www.accuweather.com",
    "https://www.adobe.com",
    "https://www.ahrefs.com",
    "https://www.aiindex.stanford.edu",
    "https://www.airbnb.com",
    "https://www.airtable.com",
    "https://www.akamai.com",
    "https://www.alexa.com",
    "https://www.alibaba.com",
    "https://www.aliexpress.com",
    "https://www.aljazeera.com",
    "https://www.allrecipes.com",
    "https://www.amazon.com",
    "https://www.americanexpress.com",
    "https://www.angel.co",
    "https://www.anthropic.com",
    "https://www.apnews.com",
    "https://www.applemusic.com",
    "https://www.archive.org",
    "https://www.archive.org/web",
    "https://www.artstation.com",
    "https://www.arxiv.org",
    "https://www.asana.com",
    "https://www.ask.com",
    "https://www.athletics.com",
    "https://www.audible.com",
    "https://www.australia.gov.au",
    "https://www.avvo.com",
    "https://www.aws.amazon.com",
    "https://www.axios.com",
    "https://www.azure.microsoft.com",
    "https://www.babbel.com",
    "https://www.baidu.com",
    "https://www.bankofamerica.com",
    "https://www.basecamp.com",
    "https://www.bbc.com",
    "https://www.bbcgoodfood.com",
    "https://www.bear.app",
    "https://www.behance.net",
    "https://www.bestbuy.com",
    "https://www.bigthink.com",
    "https://www.binance.com",
    "https://www.bing.com",
    "https://www.bloomberg.com",
    "https://www.bonappetit.com",
    "https://www.booking.com",
    "https://www.box.com",
    "https://www.boxing.com",
    "https://www.brain-pickings.org",
    "https://www.britannica.com",
    "https://www.buzzfeed.com",
    "https://www.calculatorsoup.com",
    "https://www.calendly.com",
    "https://www.canada.ca",
    "https://www.cancer.org",
    "https://www.canva.com",
    "https://www.capitalone.com",
    "https://www.careerbuilder.com",
    "https://www.cbsnews.com",
    "https://www.cdc.gov",
    "https://www.cell.com",
    "https://www.chase.com",
    "https://www.chowhound.com",
    "https://www.citibank.com",
    "https://www.clevelandclinic.org",
    "https://www.clickup.com",
    "https://www.climatereality.com",
    "https://www.cloud.google.com",
    "https://www.cloudflare.com",
    "https://www.clubhouse.com",
    "https://www.cnn.com",
    "https://www.coda.io",
    "https://www.codecademy.com",
    "https://www.codepen.io",
    "https://www.coinbase.com",
    "https://www.congress.gov",
    "https://www.conservation.org",
    "https://www.constantcontact.com",
    "https://www.converter.net",
    "https://www.cookinglight.com",
    "https://www.costco.com",
    "https://www.coursera.org",
    "https://www.courtlistener.com",
    "https://www.crazyegg.com",
    "https://www.crazygames.com",
    "https://www.crossref.org",
    "https://www.crunchyroll.com",
    "https://www.css-tricks.com",
    "https://www.culture.trip",
    "https://www.dailymotion.com",
    "https://www.deepl.com",
    "https://www.deepmind.com",
    "https://www.deezer.com",
    "https://www.defense.gov",
    "https://www.delish.com",
    "https://www.desmos.com",
    "https://www.dev.to",
    "https://www.developer.mozilla.org",
    "https://www.deviantart.com",
    "https://www.dice.com",
    "https://www.digitalocean.com",
    "https://www.discord.com",
    "https://www.discover.com",
    "https://www.disneyplus.com",
    "https://www.doaj.org",
    "https://www.doodle.com",
    "https://www.doordash.com",
    "https://www.dribbble.com",
    "https://www.drive.google.com",
    "https://www.dropbox.com",
    "https://www.drugs.com",
    "https://www.duckduckgo.com",
    "https://www.duolingo.com",
    "https://www.dw.com",
    "https://www.earthjustice.org",
    "https://www.ebay.com",
    "https://www.ecosia.org",
    "https://www.edx.org",
    "https://www.energy.gov",
    "https://www.epa.gov",
    "https://www.epicgames.com",
    "https://www.epicurious.com",
    "https://www.espace.library.uq.edu.au",
    "https://www.espn.com",
    "https://www.etrade.com",
    "https://www.etsy.com",
    "https://www.evernote.com",
    "https://www.everydayhealth.com",
    "https://www.expedia.com",
    "https://www.facebook.com",
    "https://www.fast.ai",
    "https://www.fastly.com",
    "https://www.fastmail.com",
    "https://www.fcc.gov",
    "https://www.fda.gov",
    "https://www.fidelity.com",
    "https://www.fifa.com",
    "https://www.figma.com",
    "https://www.findlaw.com",
    "https://www.flaticon.com",
    "https://www.flexjobs.com",
    "https://www.flipkart.com",
    "https://www.fodors.com",
    "https://www.fontawesome.com",
    "https://www.fonts.google.com",
    "https://www.food52.com",
    "https://www.foodnetwork.com",
    "https://www.forbes.com",
    "https://www.formstack.com",
    "https://www.formula1.com",
    "https://www.foxnews.com",
    "https://www.freecodecamp.org",
    "https://www.freepik.com",
    "https://www.freshdesk.com",
    "https://www.frommers.com",
    "https://www.ftc.gov",
    "https://www.funimation.com",
    "https://www.futurelearn.com",
    "https://www.gamespot.com",
    "https://www.gap.com",
    "https://www.genius.com",
    "https://www.geogebra.org",
    "https://www.gettyimages.com",
    "https://www.github.com",
    "https://www.glassdoor.com",
    "https://www.glitch.com",
    "https://www.gmail.com",
    "https://www.gog.com",
    "https://www.goodreads.com",
    "https://www.google.com",
    "https://www.google.com/analytics",
    "https://www.google.com/docs",
    "https://www.google.com/maps",
    "https://www.googleforms.com",
    "https://www.gov.uk",
    "https://www.grammarly.com",
    "https://www.greenpeace.org",
    "https://www.grubhub.com",
    "https://www.gtmetrix.com",
    "https://www.gutenberg.org",
    "https://www.hashnode.com",
    "https://www.hastebin.com",
    "https://www.hbomax.com",
    "https://www.health.harvard.edu",
    "https://www.healthline.com",
    "https://www.helpscout.com",
    "https://www.hemingwayapp.com",
    "https://www.heroku.com",
    "https://www.hey.com",
    "https://www.hired.com",
    "https://www.hm.com",
    "https://www.homedepot.com",
    "https://www.hopkinsmedicine.org",
    "https://www.hotels.com",
    "https://www.hotjar.com",
    "https://www.hotwire.com",
    "https://www.howstuffworks.com",
    "https://www.hub.docker.com",
    "https://www.hubspot.com",
    "https://www.huffpost.com",
    "https://www.huggingface.co",
    "https://www.hulu.com",
    "https://www.icloud.com/mail",
    "https://www.iconscout.com",
    "https://www.ign.com",
    "https://www.imdb.com",
    "https://www.indeed.com",
    "https://www.india.gov.in",
    "https://www.instagram.com",
    "https://www.instapage.com",
    "https://www.intercom.com",
    "https://www.investing.com",
    "https://www.invisionapp.com",
    "https://www.ipcc.ch",
    "https://www.irs.gov",
    "https://www.istock.com",
    "https://www.itch.io",
    "https://www.jotform.com",
    "https://www.jsfiddle.net",
    "https://www.jstor.org",
    "https://www.justia.com",
    "https://www.kaggle.com",
    "https://www.kayak.com",
    "https://www.khanacademy.org",
    "https://www.kongregate.com",
    "https://www.kotaku.com",
    "https://www.kraken.com",
    "https://www.last.fm",
    "https://www.law.cornell.edu",
    "https://www.law360.com",
    "https://www.leadpages.com",
    "https://www.legalzoom.com",
    "https://www.letterboxd.com",
    "https://www.librivox.org",
    "https://www.lighthouse-ci.appspot.com",
    "https://www.linkedin.com",
    "https://www.linkedin.com/jobs",
    "https://www.linode.com",
    "https://www.logseq.com",
    "https://www.lonelyplanet.com",
    "https://www.lowes.com",
    "https://www.lynda.com",
    "https://www.macys.com",
    "https://www.mail.com",
    "https://www.mailchimp.com",
    "https://www.majestic.com",
    "https://www.maps.apple.com",
    "https://www.marketwatch.com",
    "https://www.mastercard.com",
    "https://www.mastodon.social",
    "https://www.mathway.com",
    "https://www.mayoclinic.org",
    "https://www.medicalnewstoday.com",
    "https://www.medium.com",
    "https://www.medlineplus.gov",
    "https://www.metacritic.com",
    "https://www.miniclip.com",
    "https://www.miro.com",
    "https://www.mlb.com",
    "https://www.momondo.com",
    "https://www.monday.com",
    "https://www.monster.com",
    "https://www.morningstar.com",
    "https://www.moz.com",
    "https://www.mskcc.org",
    "https://www.nasa.gov",
    "https://www.nature.com",
    "https://www.nature.org",
    "https://www.nba.com",
    "https://www.nbcnews.com",
    "https://www.ncbi.nlm.nih.gov",
    "https://www.netflix.com",
    "https://www.netlify.com",
    "https://www.newegg.com",
    "https://www.newgrounds.com",
    "https://www.newsweek.com",
    "https://www.nfl.com",
    "https://www.nhl.com",
    "https://www.nih.gov",
    "https://www.nolo.com",
    "https://www.nordstrom.com",
    "https://www.notion.so",
    "https://www.npmjs.com",
    "https://www.npr.org",
    "https://www.nrdc.org",
    "https://www.nytimes.com",
    "https://www.obsidian.md",
    "https://www.office.com",
    "https://www.olympic.org",
    "https://www.onedrive.live.com",
    "https://www.onenote.com",
    "https://www.openai.com",
    "https://www.openculture.com",
    "https://www.openlearning.com",
    "https://www.openstreetmap.org",
    "https://www.opentable.com",
    "https://www.optimizely.com",
    "https://www.orbitz.com",
    "https://www.outlook.com",
    "https://www.overstock.com",
    "https://www.packagist.org",
    "https://www.pagespeed.web.dev",
    "https://www.pandora.com",
    "https://www.paperswithcode.com",
    "https://www.paramountplus.com",
    "https://www.pastebin.com",
    "https://www.paypal.com",
    "https://www.pbs.org",
    "https://www.pcgamer.com",
    "https://www.peacocktv.com",
    "https://www.pexels.com",
    "https://www.pga.com",
    "https://www.pinterest.com",
    "https://www.pixabay.com",
    "https://www.plex.tv",
    "https://www.plos.org",
    "https://www.pluralsight.com",
    "https://www.poki.com",
    "https://www.politico.com",
    "https://www.polygon.com",
    "https://www.postmates.com",
    "https://www.priceline.com",
    "https://www.primevideo.com",
    "https://www.protonmail.com",
    "https://www.pubmed.ncbi.nlm.nih.gov",
    "https://www.pypi.org",
    "https://www.pytorch.org",
    "https://www.quora.com",
    "https://www.rakuten.com",
    "https://www.reddit.com",
    "https://www.remote.co",
    "https://www.remoteok.com",
    "https://www.repec.org",
    "https://www.replit.com",
    "https://www.researchgate.net",
    "https://www.reuters.com",
    "https://www.roamresearch.com",
    "https://www.robinhood.com",
    "https://www.roblox.com",
    "https://www.rocketlawyer.com",
    "https://www.rome2rio.com",
    "https://www.rottentomatoes.com",
    "https://www.rubygems.org",
    "https://www.rxlist.com",
    "https://www.salesforce.com",
    "https://www.scholar.google.com",
    "https://www.schwab.com",
    "https://www.science.org",
    "https://www.sciencedirect.com",
    "https://www.scopus.com",
    "https://www.sec.gov",
    "https://www.semanticscholar.org",
    "https://www.semrush.com",
    "https://www.seriouseats.com",
    "https://www.serpstat.com",
    "https://www.shein.com",
    "https://www.shopify.com",
    "https://www.shutterstock.com",
    "https://www.sierraclub.org",
    "https://www.signal.org",
    "https://www.similarweb.com",
    "https://www.simplyhired.com",
    "https://www.simplyrecipes.com",
    "https://www.sitechecker.pro",
    "https://www.sketch.com",
    "https://www.skillshare.com",
    "https://www.skyscanner.com",
    "https://www.slack.com",
    "https://www.slate.com",
    "https://www.smashingmagazine.com",
    "https://www.snapchat.com",
    "https://www.soundcloud.com",
    "https://www.sports.yahoo.com",
    "https://www.spotify.com",
    "https://www.springer.com",
    "https://www.square.com",
    "https://www.ssrn.com",
    "https://www.stackoverflow.com",
    "https://www.startpage.com",
    "https://www.statcounter.com",
    "https://www.state.gov",
    "https://www.store.steampowered.com",
    "https://www.stripe.com",
    "https://www.supremecourt.gov",
    "https://www.surveymonkey.com",
    "https://www.symbolab.com",
    "https://www.target.com",
    "https://www.tasteofhome.com",
    "https://www.teams.microsoft.com",
    "https://www.ted.com",
    "https://www.telegram.org",
    "https://www.tennis.com",
    "https://www.tensorflow.org",
    "https://www.theatlantic.com",
    "https://www.theguardian.com",
    "https://www.thehill.com",
    "https://www.thekitchn.com",
    "https://www.threads.net",
    "https://www.tidal.com",
    "https://www.tiktok.com",
    "https://www.time.com",
    "https://www.timeanddate.com",
    "https://www.todoist.com",
    "https://www.tools.pingdom.com",
    "https://www.tradingview.com",
    "https://www.translate.google.com",
    "https://www.trello.com",
    "https://www.tripadvisor.com",
    "https://www.tumblr.com",
    "https://www.tutanota.com",
    "https://www.twitch.tv",
    "https://www.twitter.com",
    "https://www.typeform.com",
    "https://www.ubereats.com",
    "https://www.udacity.com",
    "https://www.udemy.com",
    "https://www.uefa.com",
    "https://www.ufc.com",
    "https://www.unbounce.com",
    "https://www.unitconverters.net",
    "https://www.unsplash.com",
    "https://www.usa.gov",
    "https://www.usatoday.com",
    "https://www.uscourts.gov",
    "https://www.vanguard.com",
    "https://www.vercel.com",
    "https://www.viator.com",
    "https://www.vice.com",
    "https://www.vimeo.com",
    "https://www.visa.com",
    "https://www.vk.com",
    "https://www.vox.com",
    "https://www.vultr.com",
    "https://www.vwo.com",
    "https://www.w3schools.com",
    "https://www.walmart.com",
    "https://www.washingtonpost.com",
    "https://www.wayfair.com",
    "https://www.weather.com",
    "https://www.weather.gov",
    "https://www.web.dev",
    "https://www.webmd.com",
    "https://www.webofscience.com",
    "https://www.webpagetest.org",
    "https://www.weibo.com",
    "https://www.wellfound.com",
    "https://www.wellsfargo.com",
    "https://www.weworkremotely.com",
    "https://www.whatsapp.com",
    "https://www.when2meet.com",
    "https://www.whitehouse.gov",
    "https://www.who.int",
    "https://www.whois.domaintools.com",
    "https://www.wikibooks.org",
    "https://www.wikihow.com",
    "https://www.wikipedia.org",
    "https://www.wikiversity.org",
    "https://www.wish.com",
    "https://www.wolframalpha.com",
    "https://www.worldtimeserver.com",
    "https://www.worldwildlife.org",
    "https://www.wsj.com",
    "https://www.wufoo.com",
    "https://www.wwf.org",
    "https://www.xe.com",
    "https://www.yahoo.com",
    "https://www.yahoo.com/mail",
    "https://www.yandex.com",
    "https://www.yelp.com",
    "https://www.youtube.com",
    "https://www.yummly.com",
    "https://www.zappos.com",
    "https://www.zara.com",
    "https://www.zendesk.com",
    "https://www.ziprecruiter.com",
    "https://www.zoho.com/mail",
    "https://www.zoom.us",
]

# ======================================
# FETCH SINGLE RUN FOR ONE URL
# ======================================

async def fetch_single_run(
    session: aiohttp.ClientSession,
    url: str,
    run_id: int,
    semaphore: asyncio.Semaphore
) -> dict:

    endpoint = (
        f"https://www.googleapis.com/pagespeedonline/v5/runPagespeed"
        f"?url={url}&strategy={STRATEGY}&key={API_KEY}"
    )

    async with semaphore:
        for attempt in range(1, RETRY_ATTEMPTS + 1):
            try:
                async with session.get(
                    endpoint,
                    timeout=aiohttp.ClientTimeout(total=60)
                ) as response:

                    if response.status == 429:
                        wait = RETRY_BACKOFF * attempt
                        print(f"\n⚠ Rate limited [{url}] run {run_id}, waiting {wait}s (attempt {attempt})")
                        await asyncio.sleep(wait)
                        continue

                    if response.status != 200:
                        print(f"\n✗ HTTP {response.status} for {url} run {run_id}")
                        return {"url": url, "run_number": run_id, "error": f"HTTP {response.status}"}

                    data = await response.json()

                    if "lighthouseResult" not in data:
                        err = data.get("error", {}).get("message", "Unknown error")
                        print(f"\n✗ No lighthouseResult [{url}] run {run_id}: {err}")
                        return {"url": url, "run_number": run_id, "error": err}

                    audits = data["lighthouseResult"]["audits"]

                    return {
                        # --- Metadata ---
                        "url":        url,
                        "run_number": run_id,
                        "strategy":   STRATEGY,
                        "timestamp":  time.time(),

                        # --- Raw Metrics ---
                        "performance_score": round(
                            data["lighthouseResult"]["categories"]["performance"]["score"] * 100, 1
                        ),
                        "fcp_ms":         audits.get("first-contentful-paint",   {}).get("numericValue"),
                        "lcp_ms":         audits.get("largest-contentful-paint", {}).get("numericValue"),
                        "speed_index_ms": audits.get("speed-index",              {}).get("numericValue"),
                        "ttfb_ms":        audits.get("server-response-time",     {}).get("numericValue"),
                        "tbt_ms":         audits.get("total-blocking-time",      {}).get("numericValue"),
                        "cls":            audits.get("cumulative-layout-shift",  {}).get("numericValue"),

                        "error": None,
                    }

            except asyncio.TimeoutError:
                print(f"\n⚠ Timeout [{url}] run {run_id} (attempt {attempt})")
                await asyncio.sleep(RETRY_BACKOFF * attempt)

            except Exception as e:
                print(f"\n✗ Exception [{url}] run {run_id}: {e}")
                return {"url": url, "run_number": run_id, "error": str(e)}

        return {"url": url, "run_number": run_id, "error": f"Failed after {RETRY_ATTEMPTS} attempts"}


# ======================================
# MAIN RUNNER
# ======================================

async def run(urls: list) -> tuple:

    semaphore   = asyncio.Semaphore(MAX_CONCURRENT)
    start       = time.time()
    total_tasks = len(urls) * RUNS_PER_URL

    print(f"Total requests to make: {total_tasks}  ({len(urls)} URLs × {RUNS_PER_URL} runs)\n")

    async with aiohttp.ClientSession() as session:
        tasks = [
            fetch_single_run(session, url, run_id, semaphore)
            for url    in urls
            for run_id in range(1, RUNS_PER_URL + 1)
        ]
        raw_results = await tqdm_asyncio.gather(*tasks, desc="Collecting runs")

    elapsed     = time.time() - start
    raw_results = [r for r in raw_results if r]

    print(f"\n✓ Done in {elapsed:.1f}s  |  {len(raw_results)}/{total_tasks} runs collected")

    # ======================================
    # SPLIT SUCCESS vs FAILED
    # ======================================

    raw_df  = pd.DataFrame(raw_results)
    failed  = raw_df[raw_df["error"].notna()].copy()
    success = raw_df[raw_df["error"].isna()].drop(columns=["error"]).copy()

    if not failed.empty:
        print(f"\n⚠ Failed runs ({len(failed)}):")
        print(failed[["url", "run_number", "error"]].to_string(index=False))

    # ======================================
    # AGGREGATE: mean + std per URL
    # ======================================

    metric_cols = [
        "performance_score",
        "fcp_ms",
        "lcp_ms",
        "speed_index_ms",
        "ttfb_ms",
        "tbt_ms",
        "cls",
    ]

    agg_df = (
        success
        .groupby("url")[metric_cols]
        .agg(["mean", "std"])
        .round(3)
    )

    agg_df.columns           = ["_".join(col) for col in agg_df.columns]
    agg_df["runs_collected"] = success.groupby("url")["run_number"].count()
    agg_df["strategy"]       = STRATEGY
    agg_df = agg_df.reset_index()

    return success, agg_df, failed


# ======================================
# ENTRY POINT — Colab/Jupyter compatible
# ======================================

raw_df, agg_df, failed_df = await run(urls)

# ======================================
# PREVIEW
# ======================================

print("\n--- RAW DATA (first 5 rows) ---")
print(raw_df.head().to_string(index=False))

print("\n--- AGGREGATED DATA (mean + std per URL) ---")
print(agg_df.head().to_string(index=False))

# ======================================
# SAVE CSVs
# ======================================

raw_df.to_csv("website_metrics_raw.csv",        index=False)
agg_df.to_csv("website_metrics_aggregated.csv", index=False)
failed_df.to_csv("website_metrics_failed.csv",  index=False)

print("\n✓ Saved: website_metrics_raw.csv         (all individual runs)")
print("✓ Saved: website_metrics_aggregated.csv  (mean + std per URL)")
print("✓ Saved: website_metrics_failed.csv      (failed runs)")


Total requests to make: 1437  (479 URLs × 3 runs)




⚠ Timeout [https://www.priceline.com] run 2 (attempt 1)

⚠ Timeout [https://www.priceline.com] run 3 (attempt 1)

⚠ Timeout [https://www.priceline.com] run 1 (attempt 1)



✗ HTTP 400 for https://www.repec.org run 2

✗ HTTP 400 for https://www.repec.org run 1

✗ HTTP 400 for https://www.repec.org run 3



✗ Exception [https://www.reuters.com] run 2: unsupported operand type(s) for *: 'NoneType' and 'int'

✗ Exception [https://www.reuters.com] run 3: unsupported operand type(s) for *: 'NoneType' and 'int'

✗ Exception [https://www.reuters.com] run 1: unsupported operand type(s) for *: 'NoneType' and 'int'



⚠ Timeout [https://www.replit.com] run 1 (attempt 1)


...


✗ HTTP 500 for https://www.funimation.com run 3

✗ HTTP 500 for https://www.funimation.com run 1

✗ HTTP 500 for https://www.funimation.com run 2



⚠ Timeout [https://www.gap.com] run 1 (attempt 1)

⚠ Timeout [https://www.gap.com] run 2 (attempt 1)



⚠ Timeout [https://www.genius.com] run 1 (attempt 1)



⚠ Timeout [https://www.genius.com] run 1 (attempt 2)



✓ Done in 5248.3s  |  1437/1437 runs collected

⚠ Failed runs (251):
                                 url  run_number                                                                                                                         error
               https://www.adobe.com           1                                                                                                                      HTTP 500
               https://www.adobe.com           2                                                                                                                      HTTP 500
               https://www.adobe.com           3                                                                                                                      HTTP 500
    https://www.aiindex.stanford.edu           1                                                                                                                      HTTP 400
    https://www.aiindex.stanford.edu           2       